In [4]:
import torch
from mmseg.models import build_segmentor
from mmseg.apis import init_model
from mmengine.config import Config

In [6]:
config_file = '/mnt/camobi_2/PHMG/mmsegmentation/configs/segformer/segformer_mit-b5_8xb2-160k_ade20k-512x512.py'
checkpoint_file = 'checkpoints/deeplabv3_r50-d8_512x1024_80k_cityscapes_20201226_162614-162d8b47.pth'
model = init_model(config_file, device='cuda:0')  # Set 'cuda:0' if using GPU



/mnt/camobi_2/PHMG/mmsegmentation/mmseg/models/losses/cross_entropy_loss.py:250: UserWarning: Default ``avg_non_ignore`` is False, if you would like to ignore the certain label and average loss over non-ignore labels, which is the same with PyTorch official cross_entropy, set ``avg_non_ignore=True``.
  warnings.warn(


In [7]:
model

EncoderDecoder(
  (data_preprocessor): SegDataPreProcessor()
  (backbone): MixVisionTransformer(
    (layers): ModuleList(
      (0): ModuleList(
        (0): PatchEmbed(
          (projection): Conv2d(3, 64, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
          (norm): LayerNorm((64,), eps=1e-06, elementwise_affine=True)
        )
        (1): ModuleList(
          (0-2): 3 x TransformerEncoderLayer(
            (norm1): LayerNorm((64,), eps=1e-06, elementwise_affine=True)
            (attn): EfficientMultiheadAttention(
              (attn): MultiheadAttention(
                (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
              )
              (proj_drop): Dropout(p=0.0, inplace=False)
              (dropout_layer): DropPath()
              (sr): Conv2d(64, 64, kernel_size=(8, 8), stride=(8, 8))
              (norm): LayerNorm((64,), eps=1e-06, elementwise_affine=True)
            )
            (norm2): LayerNorm((64,), eps=1e-

In [64]:
birefnet(torch.rand(1, 1, 512, 512))

RuntimeError: Given groups=1, weight of size [192, 3, 4, 4], expected input[1, 1, 512, 512] to have 3 channels, but got 1 channels instead

In [36]:
torch.set_float32_matmul_precision(['high', 'highest'][0])
birefnet.to('cuda:0')
birefnet.eval()

def extract_object(birefnet, imagepath):
    # Data settings
    image_size = (1024, 1024)
    transform_image = transforms.Compose([
        transforms.Resize(image_size),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    image = Image.open(imagepath)
    input_images = transform_image(image).unsqueeze(0).to('cuda')

    # Prediction
    with torch.no_grad():
        preds = birefnet(input_images)[-1].sigmoid().cpu()
    pred = preds[0].squeeze()
    pred_pil = transforms.ToPILImage()(pred)
    mask = pred_pil.resize(image.size)
    image.putalpha(mask)
    return image, mask

# Visualization
plt.axis("off")
test = extract_object(birefnet, imagepath='a-male-singer-on-stage-playing-guitar-EGW2H5.jpg')
plt.imshow(extract_object(birefnet, imagepath='a-male-singer-on-stage-playing-guitar-EGW2H5.jpg')[0])
plt.show()


PromptEncoder(
  (pe_layer): PositionEmbeddingRandom()
  (point_embeddings): ModuleList(
    (0-3): 4 x Embedding(1, 256)
  )
  (not_a_point_embed): Embedding(1, 256)
  (mask_downscaling): Sequential(
    (0): Conv2d(1, 4, kernel_size=(2, 2), stride=(2, 2))
    (1): LayerNorm2d()
    (2): GELU(approximate='none')
    (3): Conv2d(4, 16, kernel_size=(2, 2), stride=(2, 2))
    (4): LayerNorm2d()
    (5): GELU(approximate='none')
    (6): Conv2d(16, 256, kernel_size=(1, 1), stride=(1, 1))
  )
  (no_mask_embed): Embedding(1, 256)
)

In [39]:
predictor.get_image_embedding().shape

torch.Size([2, 256, 64, 64])

In [43]:
input_point = None
input_label = None
mask_input, unnorm_coords, labels, unnorm_box = predictor._prep_prompts(input_point, input_label, box=None, mask_logits=None, normalize_coords=True)
sparse_embeddings, dense_embeddings = predictor.model.sam_prompt_encoder(points=(unnorm_coords, labels),boxes=None,masks=None,)

high_res_features = [feat_level[-1].unsqueeze(0) for feat_level in predictor._features["high_res_feats"]]
low_res_masks, prd_scores, _, _ = predictor.model.sam_mask_decoder(image_embeddings=predictor._features["image_embed"],image_pe=predictor.model.sam_prompt_encoder.get_dense_pe(),sparse_prompt_embeddings=sparse_embeddings,dense_prompt_embeddings=dense_embeddings,multimask_output=True,repeat_image=False,high_res_features=high_res_features,)
prd_masks = predictor._transforms.postprocess_masks(low_res_masks, predictor._orig_hw[-1])# Upscale the masks to the original image resolution


AttributeError: 'NoneType' object has no attribute 'shape'

In [46]:
high_res_features = [feat_level[-1].unsqueeze(0) for feat_level in predictor._features["high_res_feats"]]

# Run mask decoding using only image embeddings
low_res_masks, prd_scores, _, _ = predictor.model.sam_mask_decoder(
    image_embeddings=predictor._features["image_embed"],  # Use image embeddings directly
    image_pe=predictor.model.sam_prompt_encoder.get_dense_pe(),  # Positional encoding
    # sparse_prompt_embeddings=None,  # No sparse prompt embeddings
    # dense_prompt_embeddings=None,   # No dense prompt embeddings
    multimask_output=False,
    repeat_image=False,
    high_res_features=high_res_features,
)

# Upscale the masks to the original image resolution
prd_masks = predictor._transforms.postprocess_masks(low_res_masks, predictor._orig_hw[-1])

TypeError: MaskDecoder.forward() missing 2 required positional arguments: 'sparse_prompt_embeddings' and 'dense_prompt_embeddings'